# EXP_050C_truecrossattn_logcosh — True Cross-Attention Re-evaluation

**Mục đích:** Đánh giá lại cấu hình tốt nhất (Swin-B + PhoBERT + Cross-Attention + Log-Cosh) sau khi sửa lỗi suy biến cross-attention (1×1 → token↔patch thật) và sửa lỗi unfreeze silent-skip cho Swin-B.

| Thành phần | Cấu hình | Ghi chú |
|---|---|---|
| Image Backbone | Swin-B | `swin_base_patch4_window7_224` |
| Text Backbone | PhoBERT | `vinai/phobert-base-v2` |
| Fusion | **True Cross-Attention (token↔patch)** | Đã refactor, không còn degenerate 1×1 |
| Loss | Log-Cosh | |
| Seed | 42 | |

**Hyperparameters (tối ưu cho T4/L4 11GB):**
- `--batch_size 32 --grad_accum_steps 1` (effective batch = 32, nằm trong vùng tối ưu 8-32 theo literature)
- `--lr 1e-5 --warmup_ratio 0.1` (giữ nguyên, không cần scale vì 1e-5 đã thấp)
- `--unfreeze_text_layers 1 --unfreeze_image_layers 1` (giờ thật sự unfreeze sau fix)
- `--use_amp` (bắt buộc cho VRAM 11GB)

---

## ⚠️ Baseline để so sánh (số cũ, EXP_050C gốc)

Số cũ được train với cross-attention **degenerate** (1×1) và unfreeze **silent-skip** cho Swin-B:

| Metric | Số cũ (EXP_050C) |
|---|---:|
| mean_mae | **1.1080** |
| overall_mae | **0.9130** |
| aspect_mae | 1.1567 |
| r2_overall | 0.6312 |

**Quyết định:**
- Nếu số mới **≤** số cũ → dùng checkpoint mới (XAI attention viz có ý nghĩa).
- Nếu số mới **>** số cũ → giữ checkpoint cũ, bỏ notebook này (XAI vẫn dùng ckpt cũ, note cross-attn).

---

## ⚠️ Cảnh báo VRAM

Cross-attention mới (token↔patch) tốn VRAM **cao hơn** version cũ (degenerate) nhiều:
- Cũ: attention `[B, 8, 1, 1]` → negligible
- Mới: attention `[B, 8, 256, 49]` × 2 chiều + 49 patch Swin forward

**Nếu OOM trên batch 32:**
1. Giảm `--batch_size` xuống 16, tăng `--grad_accum_steps` lên 2 (effective batch = 32, số gần như không đổi).
2. Nếu vẫn OOM: `--batch_size 8 --grad_accum_steps 4`.
3. Nếu vẫn OOM: `--batch_size 4 --grad_accum_steps 8` + `--max_length 128`.

Chạy 1 epoch test trước khi commit full 15 epoch.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

> **Quan trọng:** Notebook này clone từ `main` branch để lấy code cross-attention đã refactor. Nếu chạy trên branch khác, kết quả sẽ sai.

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!git log --oneline -1  # verify latest commit (should have 'refactor codebase')
!pip install -r requirements.txt -q

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configure paths

Load pretrained weights từ Phase 2 (Swin-B) và Phase 3 (PhoBERT) — **giống EXP_050C gốc**.

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_050C_truecrossattn_logcosh'

BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'   # Phase 2 winner
BEST_TEXT_MODEL   = 'vinai/phobert-base-v2'           # Phase 3 winner
BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'
BEST_TEXT_EXP_ID  = 'EXP_030B_bestimage_phobert_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')
print(f'Fusion: cross_attention (TRUE token<->patch) | Image: {BEST_IMAGE_MODEL} | Text: {BEST_TEXT_MODEL}')

### STEP 5: Load pretrained weights

Checkpoint text/image từ Phase 2/3 là **unimodal** (`train_text`/`train_image`), không dính refactor cross-attention → dùng lại được.

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')

### STEP 6: Train Fusion (True Cross-Attention + Log-Cosh)

**Cấu hình VRAM-optimized (T4/L4 11GB):**
- `--batch_size 32 --grad_accum_steps 1` (effective batch = 32)
- `--unfreeze_text_layers 1 --unfreeze_image_layers 1` (thật sự unfreeze sau fix)
- `--use_amp` (bắt buộc)

> **Nếu OOM:** xem STEP 6 fallback ở dưới.

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type cross_attention \
  --text_model_name {BEST_TEXT_MODEL} \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 15 \
  --batch_size 32 \
  --grad_accum_steps 1 \
  --lr 1e-5 \
  --patience 5 \
  --loss_fn logcosh \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id {EXP_ID} \
  --exp_dir ./experiments

#### Fallback nếu OOM ở STEP 6

Bỏ comment 1 trong các cell dưới theo thứ tự (batch giảm dần, effective batch giữ = 32):

In [ ]:
# # Fallback 1: batch 16 + grad_accum 2
# !python main.py \
#   --mode train_fusion \
#   --fusion_type cross_attention \
#   --text_model_name {BEST_TEXT_MODEL} \
#   --image_model_name {BEST_IMAGE_MODEL} \
#   --epochs 15 --batch_size 16 --grad_accum_steps 2 \
#   --lr 1e-5 --patience 5 --loss_fn logcosh \
#   --unfreeze_text_layers 1 --unfreeze_image_layers 1 \
#   --seed 42 --use_amp --exp_id {EXP_ID} --exp_dir ./experiments

In [ ]:
# # Fallback 2: batch 8 + grad_accum 4
# !python main.py \
#   --mode train_fusion \
#   --fusion_type cross_attention \
#   --text_model_name {BEST_TEXT_MODEL} \
#   --image_model_name {BEST_IMAGE_MODEL} \
#   --epochs 15 --batch_size 8 --grad_accum_steps 4 \
#   --lr 1e-5 --patience 5 --loss_fn logcosh \
#   --unfreeze_text_layers 1 --unfreeze_image_layers 1 \
#   --seed 42 --use_amp --exp_id {EXP_ID} --exp_dir ./experiments

In [ ]:
# # Fallback 3 (cùng spirit số cũ - frozen, không unfreeze):
# # batch 4 + grad_accum 8 + max_length 128 + frozen
# !python main.py \
#   --mode train_fusion \
#   --fusion_type cross_attention \
#   --text_model_name {BEST_TEXT_MODEL} \
#   --image_model_name {BEST_IMAGE_MODEL} \
#   --epochs 15 --batch_size 4 --grad_accum_steps 8 \
#   --max_length 128 \
#   --lr 1e-5 --patience 5 --loss_fn logcosh \
#   --unfreeze_text_layers 0 --unfreeze_image_layers 0 \
#   --seed 42 --use_amp --exp_id {EXP_ID} --exp_dir ./experiments

### STEP 7: Evaluate on Test Set

In [ ]:
!python test.py \
  --mode train_fusion \
  --fusion_type cross_attention \
  --text_model_name {BEST_TEXT_MODEL} \
  --image_model_name {BEST_IMAGE_MODEL} \
  --loss_fn logcosh \
  --exp_id {EXP_ID} \
  --exp_dir ./experiments \
  --save_path ./experiments/{EXP_ID}

### STEP 8: Save to Drive + so sánh với số cũ

In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

# --- VALIDATION METRICS (mới) ---
with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results (Validation) ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

In [ ]:
# --- TEST METRICS (mới) ---
test_path = f'./experiments/{EXP_ID}/test_metrics.json'
try:
    with open(test_path) as f:
        t = json.load(f)
    print(f'\n=== {EXP_ID} Results (Test) ===')
    print()
    print("             MAE      RMSE      R2")
    print(f"  food     : {t['mae_food']:.4f}   {t['rmse_food']:.4f}   {t['r2_food']:.4f}")
    print(f"  price    : {t['mae_price']:.4f}   {t['rmse_price']:.4f}   {t['r2_price']:.4f}")
    print(f"  atmos    : {t['mae_atmos']:.4f}   {t['rmse_atmos']:.4f}   {t['r2_atmos']:.4f}")
    print(f"  service  : {t['mae_service']:.4f}   {t['rmse_service']:.4f}   {t['r2_service']:.4f}")
    print(f"  overall  : {t['mae_overall']:.4f}   {t['rmse_overall']:.4f}   {t['r2_overall']:.4f}")
    print()
    print(f"  mean_mae   : {t['mean_mae']:.4f}")
    print(f"  aspect_mae : {t['aspect_mae']:.4f}")
    print(f"  overall_mae: {t['overall_mae']:.4f}")
except FileNotFoundError:
    print(f'(test_metrics.json not found at {test_path} — có thể test chưa chạy xong)')

In [ ]:
# ============================================================
# SO SÁNH SIDE-BY-SIDE: số mới (true cross-attn) vs số cũ (degenerate)
# ============================================================

# Số cũ (đã commit, từ metrics_EXP_050C_bestfusion_logcosh.json)
OLD = {
    'mean_mae': 1.1080,
    'overall_mae': 0.9130,
    'aspect_mae': 1.1567,
    'r2_overall': 0.6312,
}

print('='*60)
print('SO SÁNH: TRUE CROSS-ATTN (mới) vs DEGENERATE (cũ)')
print('='*60)
print(f"{'Metric':<18} {'Cũ (degen)':>12} {'Mới (true)':>12} {'Δ':>10} {'Đánh giá':>12}")
print('-'*60)

for key, old_val in OLD.items():
    new_val = m.get(key, None)  # dùng val metrics (m)
    if new_val is not None:
        delta = new_val - old_val
        # lower is better cho mae/aspect/mean; higher is better cho r2
        if 'r2' in key:
            better = '✅ tốt hơn' if delta > 0 else ('❌ tệ hơn' if delta < 0 else '=')
        else:
            better = '✅ tốt hơn' if delta < 0 else ('❌ tệ hơn' if delta > 0 else '=')
        print(f"{key:<18} {old_val:>12.4f} {new_val:>12.4f} {delta:>+10.4f} {better:>12}")
    else:
        print(f"{key:<18} {old_val:>12.4f} {'N/A':>12} {'N/A':>10} {'N/A':>12}")

print('-'*60)
print()
new_mean = m.get('mean_mae', 999)
new_overall = m.get('overall_mae', 999)
if new_mean <= OLD['mean_mae'] and new_overall <= OLD['overall_mae']:
    print('🎉 KẾT LUẬN: Số mới TỐT HƠN hoặc NGANG → dùng checkpoint mới cho XAI.')
    print('   → Copy best_model_train_fusion.pth sang EXP_060A để XAI attention viz có ý nghĩa.')
else:
    print('⚠️  KẾT LUẬN: Số mới TỆ HƠN → giữ checkpoint cũ (EXP_050C gốc).')
    print('   → XAI vẫn dùng ckpt cũ, cross-attn viz sẽ dùng note "attention degenerate".')
    print('   → Báo cáo: true cross-attn chưa vượt degenerate trên dataset 4800 mẫu (khớp literature)')

### STEP 9 (tùy chọn): Nếu số mới tốt hơn → copy checkpoint sang EXP_060A

Chỉ chạy cell dưới nếu STEP 8 kết luận "dùng checkpoint mới".

In [ ]:
# # Chỉ bỏ comment nếu số mới TỐT HƠN
# import shutil, os
# SRC = f'./experiments/{EXP_ID}/best_model_train_fusion.pth'
# DST_DIR = f'{DRIVE_ROOT}/experiments/EXP_060A_bestsequential_full_configuration'
# os.makedirs(DST_DIR, exist_ok=True)
# shutil.copy(SRC, f'{DST_DIR}/best_model_train_fusion.pth')
# # copy luôn metrics + config để XAI load_model đọc đúng
# for fn in ['metrics.json', 'config.yaml', 'config.json']:
#     p = f'./experiments/{EXP_ID}/{fn}'
#     if os.path.exists(p):
#         shutil.copy(p, f'{DST_DIR}/{fn}')
# print(f'Copied {SRC} → {DST_DIR}/best_model_train_fusion.pth')
# print('XAI giờ sẽ load checkpoint true cross-attention → attention viz có ý nghĩa.')